# KNN

In [ ]:
import numpy as np
import seaborn
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
def import_data():
    with open("phishing.data", 'r') as f:
        data = f.read().splitlines()
    data = [list(map(int, line.split(','))) for line in data]
    X = np.array([line[:-1] for line in data])
    y = np.array([line[-1] for line in data])

    return X, y

## distance metric

In [ ]:
def dist(x_1,x_2):
    res = x_1 - x_2
    res = np.array(res)
    return np.dot(res, res)

## classify

In [ ]:
def classify_fast(X_train, y_train, X_to_predict, metric="euclidean", p=3):
    train_labels = np.asarray(y_train)
    train_features = np.asarray(X_train)
    predict_features = np.asarray(X_to_predict)

    diff = predict_features[:, np.newaxis] - train_features


    if metric == "euclidean":  
        dists = np.sqrt(np.sum(diff**2, axis=2))
        
    elif metric == "manhattan":  
        dists = np.sum(np.abs(diff), axis=2)
        
    elif metric == "chebyshev":  
        dists = np.max(np.abs(diff), axis=2)
        
    elif metric == "minkowski": 
        dists = np.sum(np.abs(diff)**p, axis=2)**(1/p)
        
    else:
        raise ValueError(f"Nieznana metryka: {metric}")
    
    nearest_indices = np.argsort(dists, axis=1)
    return nearest_indices, train_labels

In [ ]:
# to utils:
def get_mode(row):
    shift = 1 if np.min(row) < 0 else 0
    counts = np.bincount((row + shift).astype(int))
    return np.argmax(counts) - shift

## check

In [ ]:
X,y = import_data()
X_test = []
X_val = []
X_train = []
y_val = []
y_test = []
y_train = []



for i in range(len(X)):
    num = np.random.rand()
    if num < 0.6:
        X_train.append(X[i])
        y_train.append(y[i])
    elif num < 0.8:
        X_val.append(X[i])
        y_val.append(y[i])
    else:
        X_test.append(X[i])
        y_test.append(y[i])
k_values = [1,3,5,7,9,11,13,15,17,19,21]
metrics = ["euclidean", "manhattan", "chebyshev","minkowski"]
best_accuracy = 0
best_k = None
best_metric = None


for metric in metrics:
    accuracies = []
    print(f"Evaluating metric: {metric}")
    val_indices, train_labels = classify_fast(X_train, y_train, X_val, metric=metric)
    for k in k_values:
        # Wybieramy k pierwszych sąsiadów
        neighbor_labels = train_labels[val_indices[:, :k]]
        y_pred_val = np.apply_along_axis(get_mode, 1, neighbor_labels)

        accuracy_val = np.mean(y_pred_val == y_val)

        accuracies.append(accuracy_val)
        print(f"for k={k}: Accuracy: {accuracy_val*100:.2f}%")
        if accuracy_val > best_accuracy:
            best_accuracy = accuracy_val
            best_k = k
            best_metric = metric

    plt.plot(k_values, accuracies, marker='o', label=metric)
    plt.title(f'Dokładność dla różnych wartości k dla różnych metryk')
    plt.xlabel('k')
    plt.ylabel('Dokładność')
    plt.legend()

_,y_pred = classify_fast(X_train, y_train, X_test, metric=best_metric)
print(f"Best k: {best_k}, Best Accuracy: {best_accuracy*100:.2f}%")
print(f"Best metric: {best_metric}")
y_pred = np.asarray(y_pred)
y_test = np.asarray(y_test)
def get_confusion_matrix(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == -1) & (y_pred == -1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))
    return tp, tn, fp, fn

nearest_indices, train_labels = classify_fast(X_train, y_train, X_test, metric=best_metric)
best_neighbor_labels = train_labels[nearest_indices[:, :best_k]]
y_pred = np.apply_along_axis(get_mode, 1, best_neighbor_labels)

y_pred = np.asarray(y_pred)
y_test = np.asarray(y_test)

tp, tn, fp, fn = get_confusion_matrix(y_test, y_pred)
print(f"test size: {len(y_test)}")
print("Confusion Matrix:")
print(f"TP: {tp} | FP: {fp}")
print(f"FN: {fn} | TN: {tn}")

: 